<p align="center">
  <img src="https://raw.githubusercontent.com/CSSB-SNU/Thal-Kak/main/assets/thalkak_w_plang.png" height="200" style="vertical-align: middle;">
</p>

End-to-end protein structure prediction: **MSA** (ColabFold) → **structure** (Boltz-2 / Chai-1 / Protenix / ESMFold2) → **OpenMM relaxation**, with per-model confidence-based top-5 selection.

**How to run:** set your inputs in step 1, then `Runtime` → `Run all`.

> **Requirements**
> - A **GPU** runtime: `Runtime` → `Change runtime type` → **GPU** (T4 is fine).
> - Protein targets only (RNA needs a large local database built offline).
> - Environment install is fast (**~3 min** via pixi); the first run also downloads model weights (a few minutes).
> - **Memory:** the standard runtime has ~12 GB RAM. Chai-1, Protenix, and ESMFold2 load a large ESM model, so multi-chain or otherwise large inputs can exhaust RAM and crash the kernel. For those, switch to a **High-RAM runtime** (`Runtime` → `Change runtime type`) or reduce the workload (fewer seeds, smaller inputs). Colab does not permit adding swap.

In [1]:
#@title 1. Input sequence & options
import os

jobname = "exp." #@param {type:"string"}
#@markdown - Protein sequence. For a **complex**, separate chains with `:`.
query_sequence = "MFRARDEDGIINPDKPVLSEKPSRTAALEASRFSHSSGKHMWKASGVAHPSPCPGDAGMRQKKKRPHEDTVRSCHL" #@param {type:"string"}
#@markdown - Stoichiometry, e.g. `A1` (monomer), `A2` (homodimer), `A1B1`
#@markdown   (heterodimer). Leave as `UNK` to use one copy of each chain.
stoichiometry = "UNK" #@param {type:"string"}
#@markdown ---
#@markdown ### Structure models
#@markdown Tick every model you want. They run in order against one MSA,
#@markdown so comparing several costs a single alignment search.
boltz2 = True #@param {type:"boolean"}
chai1 = False #@param {type:"boolean"}
protenix_v1 = False #@param {type:"boolean"}
protenix_v2 = False #@param {type:"boolean"}
esmfold2 = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### Run options
relax_method = "openmm" #@param ["openmm", "none"]
num_seeds = 2 #@param {type:"integer"}
#@markdown - Confidence metric that ranks the top-5 models.
top5_metric = "ranking_score" #@param ["ranking_score", "ptm", "iptm", "plddt"]
#@markdown ---
#@markdown ### Google Drive cache (optional)
#@markdown Cache the ticked models' large checkpoint file(s) on your Drive
#@markdown so later sessions skip those (large) downloads. The first successful
#@markdown run fills it; later runs read the checkpoint straight from Drive.
#@markdown Small files still download each run. Uncheck to skip caching.
use_drive_cache = True #@param {type:"boolean"}
drive_cache_dir = "/content/drive/MyDrive/thalkak_cache" #@param {type:"string"}

# The ticked models, in the order Thal-Kak will run them.
structure_models = [m for m, on in (("boltz2", boltz2), ("chai1", chai1),
                                    ("protenix_v1", protenix_v1),
                                    ("protenix_v2", protenix_v2),
                                    ("esmfold2", esmfold2)) if on]
assert structure_models, "Tick at least one structure model."

# Build a full-mode input yaml (Method + Entity) for `thalkak full --input`.
import re
seqs = [s.strip().upper().replace(" ", "")
        for s in query_sequence.split(":") if s.strip()]
assert seqs, "query_sequence is empty"
workdir = f"/content/{jobname}"
os.makedirs(workdir, exist_ok=True)
# Map stoichiometry (e.g. "A2B1") to a per-chain copy count; "UNK" -> 1 each.
if stoichiometry.strip().upper() == "UNK":
    copies = [1] * len(seqs)
else:
    _counts = dict(re.findall(r"([A-Za-z])(\d+)", stoichiometry))
    copies = [int(_counts.get(chr(65 + i), 1)) for i in range(len(seqs))]
input_yaml = f"{workdir}/input.yaml"
with open(input_yaml, "w") as f:
    f.write("Method:\n")
    f.write(f"  jobname: {jobname}\n")
    f.write("  msa: colab\n")
    f.write(f"  structure: [{', '.join(structure_models)}]\n")
    f.write(f"  relax: {relax_method}\n")
    f.write(f"  top5_metric: {top5_metric}\n")
    f.write(f"  n_seed: {int(num_seeds)}\n")
    f.write("Entity:\n")
    for i, s in enumerate(seqs):
        f.write("  - type: protein\n")
        f.write(f"    seq: {s}\n")
        f.write(f"    copy: {copies[i]}\n")

print(f"Wrote input yaml ({len(seqs)} chain(s)) to {input_yaml}")
print(f"models={', '.join(structure_models)}  relax={relax_method}  "
      f"seeds={num_seeds}  stoi={stoichiometry}")

# Weight cache for the ticked models only. Everything downloads to local
# disk; only large checkpoint files are mirrored to Drive (as themselves,
# not a tar). Small files -- notably boltz's ~45k CCD mol pickles -- always
# download locally, so they never clutter Drive or hit its FUSE small-file
# limits. A cached checkpoint is symlinked back in and read straight from Drive.
LOCAL_CACHE = "/content/thalkak_cache"
BIG_FILE_BYTES = 50 * 1024 * 1024   # a file this big or larger is a checkpoint
_MODEL_CACHE = {"boltz2": ("BOLTZ_CACHE", "boltz"),
                "chai1": ("CHAI_DOWNLOADS_DIR", "chai"),
                "protenix_v1": ("PROTENIX_CHECKPOINT_DIR", "protenix"),
                "protenix_v2": ("PROTENIX_CHECKPOINT_DIR", "protenix"),
                "esmfold2": ("HF_HOME", "hf")}
# One cache dir per backend, so several models can be cached side by side.
# protenix_v1 and protenix_v2 share theirs, hence the de-duplication.
cache_targets = []
for _m in structure_models:
    _t = _MODEL_CACHE[_m]
    if _t not in cache_targets:
        cache_targets.append(_t)
drive_pairs = []   # (local dir, Drive dir) per backend; cell 4 mirrors into these
if use_drive_cache:
    from google.colab import drive
    drive.mount("/content/drive")
linked = 0
for env_var, sub in cache_targets:
    local_sub = os.path.join(LOCAL_CACHE, sub)
    os.makedirs(local_sub, exist_ok=True)
    os.environ[env_var] = local_sub      # all weights download to local disk
    if not use_drive_cache:
        continue
    drive_big = os.path.join(drive_cache_dir, sub + "_ckpt")
    os.makedirs(drive_big, exist_ok=True)
    drive_pairs.append((local_sub, drive_big))
    # Symlink any cached checkpoint into the local cache so the backend sees
    # it as present (skips that download) and reads it directly from Drive.
    for _root, _dirs, _files in os.walk(drive_big):
        for _fn in _files:
            _src = os.path.join(_root, _fn)
            _dst = os.path.join(local_sub, os.path.relpath(_src, drive_big))
            os.makedirs(os.path.dirname(_dst), exist_ok=True)
            if not os.path.lexists(_dst):
                os.symlink(_src, _dst); linked += 1
if use_drive_cache:
    print(f"weights: {linked} checkpoint file(s) linked from Drive; any "
          f"remaining files download locally.")
else:
    print("Drive cache off; weights download for this session only.")

Wrote input yaml (1 chain(s)) to /content/exp./input.yaml
models=boltz2  relax=openmm  seeds=2  stoi=UNK
Mounted at /content/drive
weights: 0 checkpoint file(s) linked from Drive; any remaining files download locally.


In [2]:
#@title 2. Clone the Thal-Kak repository
import os, subprocess

REPO = "CSSB-SNU/Thal-Kak"
REPO_DIR = "/content/Thal-Kak"

if not os.path.isdir(REPO_DIR):
    rc = subprocess.run(
        ["git", "clone", "--depth", "1",
         f"https://github.com/{REPO}.git", REPO_DIR]
    ).returncode
    if rc != 0:
        raise RuntimeError("git clone failed.")
    print("Cloned", REPO)
else:
    print("Repo already present, skipping clone.")

Cloned CSSB-SNU/Thal-Kak


In [3]:
#@title 3. Install dependencies (pixi) — ~3 min
#@markdown Installs the [pixi](https://pixi.sh) package manager, resolves the
#@markdown locked `thalkak` environment from `pixi.lock` (no dependency solve),
#@markdown and applies the ColabFold templates patch. Re-running is cheap.
import os, shlex

script = r"""
set -e
export PATH="$HOME/.pixi/bin:$PATH"
if ! command -v pixi >/dev/null 2>&1; then
  echo "== Installing pixi =="
  curl -fsSL https://pixi.sh/install.sh | bash
  export PATH="$HOME/.pixi/bin:$PATH"
fi
cd /content/Thal-Kak
echo "== Creating the thalkak env from pixi.lock =="
pixi install
echo "== Post-install (ColabFold templates patch) =="
pixi run postinstall
echo "== Install step complete =="
"""

rc = os.system("bash -c " + shlex.quote(script))
if rc != 0:
    raise RuntimeError("Installation failed — see the log above.")

In [4]:
#@title 4. Run Thal-Kak (MSA → structure → relax)
import subprocess, shlex

cmd = (
    'export PATH="$HOME/.pixi/bin:$PATH" && '
    "cd /content/Thal-Kak && "
    # run inside the pixi env; -u so progress prints stream live in Colab
    "PYTHONUNBUFFERED=1 pixi run python -u thalkak.py full "
    f"--input {shlex.quote(input_yaml)}"
)
# Stream stdout/stderr line by line so progress shows live in the cell.
proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError("Thal-Kak run failed — see the log above.")
print("Done. Outputs under:", workdir)

# Success: mirror any newly-downloaded large checkpoints to Drive. These are
# single big files (Drive FUSE handles them fine); the many small mol pickles
# are never copied. .part -> replace so an interrupted copy never looks done.
import os, shutil
pushed = 0
for local_sub, drive_big in globals().get("drive_pairs", []):
    for _root, _dirs, _files in os.walk(local_sub):
        for _fn in _files:
            _src = os.path.join(_root, _fn)
            if os.path.islink(_src):
                continue                       # already backed by Drive
            try:
                if os.path.getsize(_src) < BIG_FILE_BYTES:
                    continue
            except OSError:
                continue
            _rel = os.path.relpath(_src, local_sub)
            _dst = os.path.join(drive_big, _rel)
            if os.path.exists(_dst):
                continue
            os.makedirs(os.path.dirname(_dst), exist_ok=True)
            try:
                print(f"caching checkpoint {_rel} to Drive ...", flush=True)
                shutil.copyfile(_src, _dst + ".part")   # data only (FUSE-safe)
                os.replace(_dst + ".part", _dst)
                pushed += 1
            except Exception as e:
                print(f"  (skipped {_rel}: {e})", flush=True)
if pushed:
    print(f"cached {pushed} checkpoint file(s) to Drive.")

2026-09-30 06:04:45 | full      | Wrote polymer FASTA from input entities: /content/exp./exp..fa
2026-09-30 06:04:45 | full      | Running Thal-Kak full: msa=colab, structure=['boltz2'], relax=openmm, input FASTA: /content/exp./exp..fa
2026-09-30 06:04:45 | full      | ================== MSA generation (colab) ==================
2026-09-30 06:04:45 | msa       | Generating MSA...
2026-09-30 06:04:45 | msa       | Running MSA generation using Colab...
2026-09-30 06:04:45 | msa       | $ colabfold_batch --msa-only --templates /content/exp./exp_parsed.fa /content/exp./msa/colab
2026-09-30 06:04:52 | msa       | 2026-09-30 06:04:52,138 Running colabfold 1.6.1
2026-09-30 06:04:52 | msa       | 2026-09-30 06:04:52,217 WARNING: no GPU detected, will be using CPU
2026-09-30 06:05:00 | msa       | 2026-09-30 06:05:00,378 generated new fontManager
2026-09-30 06:05:04 | msa       | 2026-09-30 06:05:04,722 Found 7 citations for tools or databases
2026-09-30 06:05:04 | msa       | 2026-09-30 06:05:

In [5]:
#@title 5. View a predicted structure (per backend)
#@markdown Use the dropdowns to switch backend / rank / coloring; the view
#@markdown updates live. One backend per run that was ticked in cell 1.
import os, glob
import ipywidgets as W
from IPython.display import display
import matplotlib.pyplot as plt, matplotlib as mpl
from matplotlib.colors import Normalize, LinearSegmentedColormap
try:
    import py3Dmol
except ImportError:
    os.system("pip -q install py3Dmol"); import py3Dmol

run_dirs = sorted(glob.glob(f"{workdir}/top5/*_results_*"))
assert run_dirs, "No top5 dir found - did the run finish?"
# One dir per structure model; name each by the backend that produced it. A
# stale dir from an earlier run with different settings keeps its own entry.
runs = {os.path.basename(d).split("_results_")[0]: d for d in run_dirs}

def run_state(run_dir):
    relaxed = sorted(glob.glob(f"{run_dir}/relaxed/*"))
    ids = sorted({int(os.path.basename(p).split("_")[1].split(".")[0])
                  for p in glob.glob(f"{run_dir}/model_*.pdb")}) or [1]
    return (relaxed[0] if relaxed else None), ids

def structure_path(run_dir, n):
    relaxed_dir, _ = run_state(run_dir)
    if relaxed_dir:
        hits = glob.glob(f"{relaxed_dir}/model_{n}_*.pdb")
        if hits: return hits[0]
    p = f"{run_dir}/model_{n}.pdb"
    return p if os.path.exists(p) else None

def style(color):
    # relaxed/unrelaxed PDBs carry pLDDT in the B-factor column
    if color == "pLDDT":
        return {"cartoon": {"colorscheme": {"prop": "b", "gradient": "rwb",
                                            "min": 40, "max": 100}}}
    return {"cartoon": {"color": "spectrum"}}

run_dd = W.Dropdown(options=list(runs), value=next(iter(runs)),
                    description="Backend:")
_, _ids = run_state(runs[run_dd.value])
model_dd = W.Dropdown(options=_ids, value=_ids[0], description="Rank:")
color_dd = W.Dropdown(options=["pLDDT", "N to C (rainbow)"], value="pLDDT",
                      description="Color:")
cbar = W.Output()
view = py3Dmol.view(width=700, height=500)

def draw_colorbar(color):
    cbar.clear_output(wait=True)
    with cbar:
        fig, ax = plt.subplots(figsize=(4.5, 0.42))
        if color == "pLDDT":
            # matplotlib "RdBu": 0->red, 1->blue (matches 3Dmol "rwb")
            cmap, norm, label = plt.get_cmap("RdBu"), Normalize(40, 100), "pLDDT"
        else:
            cmap = LinearSegmentedColormap.from_list(
                "roygb", ["red", "orange", "yellow", "green", "blue"])  # 3Dmol spectrum
            norm, label = Normalize(0, 1), "N-term  ->  C-term"
        cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation="horizontal")
        cb.set_label(label)
        if color != "pLDDT":
            cb.set_ticks([])
        plt.show()

def refresh(_=None):
    p = structure_path(runs[run_dd.value], model_dd.value)
    if not p:
        return
    # mutate the existing viewer in place (no re-injection -> no blank in Colab)
    view.removeAllModels()
    view.addModel(open(p).read(), "pdb")
    view.setStyle(style(color_dd.value))
    view.zoomTo()
    view.update()
    draw_colorbar(color_dd.value)

def switch_run(_=None):
    # backends can produce different rank sets, so rebuild the rank options.
    # Detach refresh first: assigning options/value would fire it mid-update.
    _, ids = run_state(runs[run_dd.value])
    model_dd.unobserve(refresh, names="value")
    model_dd.options = ids
    model_dd.value = ids[0]
    model_dd.observe(refresh, names="value")
    refresh()

display(W.HBox([run_dd, model_dd, color_dd]))
p0 = structure_path(runs[run_dd.value], model_dd.value)
if p0:
    view.addModel(open(p0).read(), "pdb")
    view.setStyle(style(color_dd.value))
    view.zoomTo()
view.show()
display(cbar)
draw_colorbar(color_dd.value)

run_dd.observe(switch_run, names="value")
model_dd.observe(refresh, names="value")
color_dd.observe(refresh, names="value")


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Output()

In [6]:
#@title 6. Metrics (MSA depth, pAE, pLDDT, ranking, energy)
#@markdown One tab set per backend; pick the backend from the dropdown. The MSA
#@markdown tab is the same for all of them -- they share one alignment.
import os, glob, yaml, pandas as pd
from IPython.display import display, Image
import ipywidgets as W

run_dirs = sorted(glob.glob(f"{workdir}/top5/*_results_*"))
assert run_dirs, "No result dirs found - did the run finish?"
# One dir per structure model; name each by the backend that produced it.
runs = {os.path.basename(d).split("_results_")[0]: d for d in run_dirs}

def _first_png(*pats):
    for pat in pats:
        hits = sorted(glob.glob(pat))
        if hits: return hits[0]
    return None

def show_png(caption, *pats):
    png = _first_png(*pats)
    if png: print(caption); display(Image(png))
    else: print(f"{caption}: image not found.")

def show_msa_depth():
    a3ms = [p for p in glob.glob(f"{workdir}/msa/*/*.a3m") if "_env" not in p]
    if a3ms:
        depth = sum(1 for l in open(a3ms[0]) if l.startswith(">"))
        print(f"MSA depth: {depth} sequences  ({os.path.basename(a3ms[0])})")
    show_png("Coverage", f"{workdir}/msa/*/*coverage*.png")

def show_ranking(common):
    csv = glob.glob(f"{common}/*results_summary.csv")
    if not csv: print("summary csv not found"); return
    print("Top-5 models are chosen by ranking_score (descending):")
    display(pd.read_csv(csv[0]).head(10))

def show_energy(relaxed_dir):
    if not relaxed_dir: print("no relaxed dir"); return
    ep = os.path.join(relaxed_dir, "energies.yaml")
    if not os.path.exists(ep): print("energies.yaml not found (relax=none?)"); return
    display(pd.DataFrame(yaml.safe_load(open(ep)) or {}).T)

def build_tabs(run_dir):
    common = os.path.join(workdir, "structure", os.path.basename(run_dir), "common")
    relaxed_dirs = sorted(glob.glob(f"{run_dir}/relaxed/*"))
    relaxed_dir = relaxed_dirs[0] if relaxed_dirs else None
    tabs = [
        ("MSA depth", show_msa_depth),
        ("pAE",       lambda: show_png("pAE", f"{common}/*pae*.png")),
        ("pLDDT",     lambda: show_png("pLDDT", f"{common}/*plddt*.png")),
        ("Ranking",   lambda: show_ranking(common)),
        ("Energy",    lambda: show_energy(relaxed_dir)),
    ]
    outs = [W.Output() for _ in tabs]
    tab = W.Tab(children=outs)
    for i, (t, _) in enumerate(tabs): tab.set_title(i, t)
    for o, (_, fn) in zip(outs, tabs):
        with o:
            try: fn()
            except Exception as e: print("error:", e)
    return tab

run_dd = W.Dropdown(options=list(runs), value=next(iter(runs)),
                    description="Backend:")
panel = W.Output()

def render(_=None):
    panel.clear_output(wait=True)
    with panel:
        display(build_tabs(runs[run_dd.value]))

run_dd.observe(render, names="value")
display(run_dd, panel)
render()


Dropdown(description='Backend:', options=('boltz',), value='boltz')

Output()

In [7]:
#@title 7. Download results
#@markdown Zips every backend that ran (its structures, top-5, relaxed
#@markdown models, confidence logs) plus the MSA they shared, and downloads
#@markdown it. Optionally copy to Google Drive.
save_to_google_drive = True #@param {type:"boolean"}
import os, glob, zipfile

# Every backend dir, plus the one MSA all of them used.
run_dirs = sorted(glob.glob(f"{workdir}/top5/*_results_*"))
assert run_dirs, "No results found - did the run finish?"
roots = [os.path.join(workdir, "msa")]
for _d in run_dirs:
    roots += [_d, os.path.join(workdir, "structure", os.path.basename(_d))]

zip_path = f"/content/{jobname}.result.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root in roots:
        if not os.path.isdir(root): continue
        for dirpath, _, fnames in os.walk(root):
            for fn in fnames:
                full = os.path.join(dirpath, fn)
                z.write(full, os.path.relpath(full, workdir))
_names = ", ".join(os.path.basename(d).split("_results_")[0] for d in run_dirs)
print("Result archive:", zip_path, f"({len(run_dirs)} backend(s): {_names})")

if save_to_google_drive:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/{jobname}.result.zip"
    shutil.copyfile(zip_path, dst)
    print("Saved to", dst)

from google.colab import files
files.download(zip_path)

Result archive: /content/exp..result.zip (1 backend(s): boltz)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to /content/drive/MyDrive/exp..result.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes

- **Models & weights.** Each backend downloads its own weights on first run (e.g. `~/.boltz` for Boltz-2, HuggingFace cache for ESMFold2). Weights are distributed under their providers' own terms.
- **Outputs.** For a job named `insulin` the results live under `/content/insulin/`: `msa/`, `structure/`, and `top5/<job>/` with `model_1..5.pdb`, `relaxed/<method>/`, and `method_log.yaml` provenance.
- **RNA/DNA.** This notebook targets proteins. RNA MSA needs a large local database (`./install_db.sh --family rna`) that is impractical to build on Colab — run RNA targets locally instead.
- **License.** Thal-Kak is Apache-2.0; see `LICENSE` and `NOTICE` in the repo.